## Installing the SageMaker SDK

In [1]:
%pip install -q --upgrade -r requirements.txt

%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops

%pip install --no-cache-dir --force-reinstall "sagemaker>=2.245.0,<3"

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
Note: you may need to restart the kernel to use updated packages.
Found existing installation: sagemaker 2.257.6
Uninstalling sagemaker-2.257.6:
  Successfully uninstalled sagemaker-2.257.6
Found existing installation: sagemaker-core 1.0.78
Uninstalling sagemaker-core-1.0.78:
  Successfully uninstalled sagemaker-core-1.0.78
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 357.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 359.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 213.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 127.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 336.8 MB/s  0

In [1]:
import base64
import io
import json 
import shutil
import sys
import tarfile


import time 

from urllib.parse import urlparse

import requests
import boto3 
import numpy as np 
import sagemaker

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


In [9]:
%pip install -q "pyarrow>=14.0" "datasets>=2.16"

Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset
from sagemaker import image_uris
from sagemaker.huggingface import HuggingFace, HuggingFaceModel
from sagemaker.model_monitor import DataCaptureConfig 
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

In [3]:
sys.path.append('scripts')

session = sagemaker.Session()
boto_session = boto3.Session()
REGION = boto_session.region_name
ROLE = sagemaker.get_execution_role()
BUCKET = 'cxr14-tfstate-c70a9673'

print({'region': REGION, 'bucket': BUCKET, 'role': ROLE})

{'region': 'eu-west-2', 'bucket': 'cxr14-tfstate-c70a9673', 'role': 'arn:aws:iam::706317927930:role/service-role/AmazonSageMaker-ExecutionRole-20260929T205911'}


In [4]:
import matplotlib 
import matplotlib.pyplot as plt

In [5]:
from datasets import load_dataset_builder
HF_DATASET_ID   = 'BahaaEldin0/NIH-Chest-Xray-14'
b = load_dataset_builder(HF_DATASET_ID)
print(b.info.features)   # shows the real column types
print(b.info.splits)     # row counts per split

README.md:   0%|          | 0.00/710 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/73 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/73 [00:00<?, ?it/s]

{'image': Image(mode=None, decode=True), 'label': List(Value('string')), 'Patient Age': Value('int32'), 'Patient Gender': Value('string'), 'View Position': Value('string'), 'Patient ID': Value('int32')}
{'train': SplitInfo(name='train', num_bytes=36114805446.16, num_examples=89696), 'valid': SplitInfo(name='valid', num_bytes=4501036197.852, num_examples=11212), 'test': SplitInfo(name='test', num_bytes=4508370970.804, num_examples=11212)}


In [7]:
import os
import numpy as np
import pandas as pd
from huggingface_hub import HfApi
from datasets import load_dataset_builder

HF_DATASET_ID  = "BahaaEldin0/NIH-Chest-Xray-14"
LABEL_COLUMN   = "label"
PATIENT_AGE    = "Patient Age"
PATIENT_GENDER = "Patient Gender"
VIEW_POSITION  = "View Position"
PATIENT_ID     = "Patient ID"
TRAIN_SPLIT, VALID_SPLIT, TEST_SPLIT = "train", "valid", "test"

In [8]:
files = [f for f in HfApi().list_repo_files(HF_DATASET_ID, repo_type="dataset")
         if f.endswith(".parquet")]
print(len(files))
for f in files:
    print(f)

93
data/test-00000-of-00010.parquet
data/test-00001-of-00010.parquet
data/test-00002-of-00010.parquet
data/test-00003-of-00010.parquet
data/test-00004-of-00010.parquet
data/test-00005-of-00010.parquet
data/test-00006-of-00010.parquet
data/test-00007-of-00010.parquet
data/test-00008-of-00010.parquet
data/test-00009-of-00010.parquet
data/train-00000-of-00073.parquet
data/train-00001-of-00073.parquet
data/train-00002-of-00073.parquet
data/train-00003-of-00073.parquet
data/train-00004-of-00073.parquet
data/train-00005-of-00073.parquet
data/train-00006-of-00073.parquet
data/train-00007-of-00073.parquet
data/train-00008-of-00073.parquet
data/train-00009-of-00073.parquet
data/train-00010-of-00073.parquet
data/train-00011-of-00073.parquet
data/train-00012-of-00073.parquet
data/train-00013-of-00073.parquet
data/train-00014-of-00073.parquet
data/train-00015-of-00073.parquet
data/train-00016-of-00073.parquet
data/train-00017-of-00073.parquet
data/train-00018-of-00073.parquet
data/train-00019-of-0

In [10]:
%%time
df = pd.read_parquet(f"hf://datasets/{HF_DATASET_ID}/{files[0]}", columns=cols)
print(len(files), "files;", len(df), "rows in the first")

93 files; 1122 rows in the first
CPU times: user 203 ms, sys: 207 ms, total: 409 ms
Wall time: 12.3 s


In [11]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def split_of(f):
    for s in (TRAIN_SPLIT, VALID_SPLIT, TEST_SPLIT):
        if s in f:
            return s

def read_one(f):
    df = pd.read_parquet(f"hf://datasets/{HF_DATASET_ID}/{f}", columns=cols)
    df["split"] = split_of(f)
    df["file"] = f
    return df

with ThreadPoolExecutor(max_workers=8) as ex:
    parts = list(tqdm(ex.map(read_one, files), total=len(files)))

meta = pd.concat(parts, ignore_index=True)
meta.to_parquet("metadata.parquet")     # cachING : reload this next time instead of refetching
print(meta["split"].value_counts())

  0%|          | 0/93 [00:00<?, ?it/s]

split
train    89696
test     11212
valid    11212
Name: count, dtype: int64


In [12]:
os.makedirs("data", exist_ok=True)
meta["row_in_file"] = meta.groupby("file").cumcount()   # position of each row inside 
meta.to_parquet("data/metadata.parquet")

In [13]:
g = {s: set(meta.loc[meta["split"] == s, PATIENT_ID]) for s in ("train", "valid", "test")}

print("patients per split:", {s: len(v) for s, v in g.items()})
print("train ∩ valid:", len(g["train"] & g["valid"]))
print("train ∩ test: ", len(g["train"] & g["test"]))
print("valid ∩ test: ", len(g["valid"] & g["test"]))


patients per split: {'train': 27136, 'valid': 7232, 'test': 7211}
train ∩ valid: 5300
train ∩ test:  5359
valid ∩ test:  2778


In [14]:
print(meta[PATIENT_AGE].describe())
print("ages > 100:", (meta[PATIENT_AGE] > 100).sum())

print(meta[PATIENT_GENDER].value_counts())
print(meta[VIEW_POSITION].value_counts())

train = meta[meta["split"] == "train"]
label_names = sorted(set(train[LABEL_COLUMN].explode().dropna()) - {"No Finding"})
print(len(label_names), label_names)                 # expect 14
print(train[LABEL_COLUMN].explode().value_counts())

count    112120.000000
mean         46.901463
std          16.839923
min           1.000000
25%          35.000000
50%          49.000000
75%          59.000000
max         414.000000
Name: Patient Age, dtype: float64
ages > 100: 16
Patient Gender
M    63340
F    48780
Name: count, dtype: int64
View Position
PA    67310
AP    44810
Name: count, dtype: int64
14 ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']
label
No Finding            48411
Infiltration          15826
Effusion              10629
Atelectasis            9191
Nodule                 5041
Mass                   4655
Pneumothorax           4259
Consolidation          3688
Pleural_Thickening     2713
Cardiomegaly           2229
Emphysema              1968
Edema                  1851
Fibrosis               1335
Pneumonia              1120
Hernia                  186
Name: count, dtype: i

In [15]:
meta["age_clean"] = meta[PATIENT_AGE].where(meta[PATIENT_AGE].between(1, 100))   # invalid ages become NaN
print("ages set to missing:", meta["age_clean"].isna().sum())

sha = HfApi().dataset_info(HF_DATASET_ID).sha
print("dataset revision:", sha)

ages set to missing: 16
dataset revision: 932bcdba9d7d9590704d4f20bc70fc2c3a1bbad7


In [16]:
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
groups = meta[PATIENT_ID]

# 80% train+valid / 20% test, keeping each patient entirely on one side
tv_idx, te_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED).split(meta, groups=groups))
trainval = meta.iloc[tv_idx]

# split train+valid again: 12.5% of 80% is 10% overall
tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.125, random_state=SEED)
                      .split(trainval, groups=trainval[PATIENT_ID]))

meta["new_split"] = "train"
meta.loc[trainval.index[va_idx], "new_split"] = "valid"
meta.loc[meta.index[te_idx], "new_split"] = "test"

# Hard check: no patient in more than one split
n_splits_per_patient = meta.groupby(PATIENT_ID)["new_split"].nunique()
assert (n_splits_per_patient == 1).all(), "leakage!"
print(meta["new_split"].value_counts())

new_split
train    78873
test     22294
valid    10953
Name: count, dtype: int64


In [17]:
# Labels spread sensibly across the new splits?
Yall = meta[LABEL_COLUMN].map(lambda ls: [l for l in ls if l != "No Finding"])
prev = (pd.get_dummies(Yall.explode()).groupby(Yall.explode().index).max()
          .join(meta["new_split"]).groupby("new_split").sum())
print(prev[label_names].T)     # rows are findings, columns are splits

# Save ONLY the patient 
os.makedirs("configs", exist_ok=True)
(meta[[PATIENT_ID, "new_split"]].drop_duplicates()
     .sort_values(PATIENT_ID)
     .to_csv("configs/patient_split_v1.csv", index=False))

meta.to_parquet("data/metadata.parquet")   # full cache stays out of Git

new_split           test  train  valid
Atelectasis         2207   8262   1090
Cardiomegaly         618   1863    295
Consolidation        951   3280    436
Edema                454   1639    210
Effusion            2553   9533   1231
Emphysema            531   1727    258
Fibrosis             364   1145    177
Hernia                48    158     21
Infiltration        4097  13868   1929
Mass                1076   4199    507
Nodule              1280   4415    636
Pleural_Thickening   660   2424    301
Pneumonia            294    982    155
Pneumothorax        1020   3841    441
